[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/095.%20The%20Supply%20Chain%20Network%20Design%20under%20Uncertainty/P95-Tier-1_executed.ipynb)

---


# 95. Supply Chain Network Design under Uncertainty
## Tier 1: Mathematical Formulation with Integer Programming

### Purpose and scope

This notebook designs a distribution network when customer demand can take one of several scenarios with known probabilities. It chooses which facilities to open before the scenario is known, then chooses customer assignments and shipments separately for each scenario. The objective is to minimize the sum of facility-opening costs and expected transportation costs.

The input is a small network instance: facilities, customers, scenario-specific demands, facility capacities, fixed opening costs, transportation costs, and scenario probabilities. The outputs are the facility-opening decisions, scenario-specific assignments and shipments, cost summaries, visualizations, and sensitivity results. The example has three facilities, four customers, one product, and two demand scenarios.

### Model assumptions

- Scenario probabilities are known, nonnegative, and sum to one.
- Facility-opening decisions are made before demand is observed and are shared across all scenarios.
- Customer assignments may change between scenarios, but each customer is assigned to exactly one facility in each scenario. Consequently, all of that customer's demand in that scenario is supplied by its assigned facility; demand cannot be split across facilities.
- Each facility has a scenario-independent capacity, and capacity is available only when the facility is open.
- Fixed facility costs are paid once, not once per scenario. Transportation costs are proportional to shipped quantity and are included in the objective as an expectation over scenarios.
- The example has one product, indexed by \(k=0\). There are no facility-opening, transportation, inventory, shortage, or emissions decisions beyond those explicitly represented in the model.

These assumptions define a two-stage, scenario-based mixed-integer optimization model. The first-stage decisions are facility openings; the scenario-dependent assignment and shipment decisions are recourse decisions. The formulation does not represent continuous probability distributions, demand uncertainty within a scenario, or other real-world features not listed above.

### Sets, data, and decision variables

Let \(I\) be the set of facilities, \(J\) the set of customers, \(K=\{0\}\) the single-product set, and \(S\) the set of demand scenarios. For facility \(i\in I\), \(f_i\) is its fixed opening cost and \(Q_i\) its capacity. For customer \(j\in J\), product \(k\in K\), and scenario \(s\in S\), \(d_{jks}\) is demand. The probability of scenario \(s\) is \(p_s\), and \(c_{ij}\) is the per-unit transportation cost from facility \(i\) to customer \(j\).

The binary variable \(y_i\) equals one when facility \(i\) is open. The binary variable \(z_{ijs}\) equals one when customer \(j\) is assigned to facility \(i\) in scenario \(s\). The nonnegative continuous variable \(x_{ijks}\) is the quantity shipped from facility \(i\) to customer \(j\) for product \(k\) in scenario \(s\). In the code, these variables are stored in the dictionaries `y`, `z`, and `x`, respectively.

### Objective and constraints

The objective combines the one-time cost of opening facilities with the probability-weighted transportation cost in each scenario. A probability-weighted cost is an expected cost: each scenario's cost contributes in proportion to how likely that scenario is.

$$
\min_{y,z,x}\quad
\sum_{i\in I} f_i y_i
+
\sum_{s\in S} p_s\sum_{i\in I}\sum_{j\in J}\sum_{k\in K} c_{ij}x_{ijks}.
$$

The model must satisfy demand, respect facility capacity, link shipments to assignments, assign each customer to exactly one facility in each scenario, and allow assignments only to open facilities:

$$
\begin{aligned}
&\sum_{i\in I}x_{ijks}=d_{jks}
&&\forall j\in J,\ k\in K,\ s\in S,
\\
&\sum_{j\in J}x_{ij0s}\le Q_i y_i
&&\forall i\in I,\ s\in S,
\\
&x_{ijks}\le d_{jks}z_{ijs}
&&\forall i\in I,\ j\in J,\ k\in K,\ s\in S,
\\
&\sum_{i\in I}z_{ijs}=1
&&\forall j\in J,\ s\in S,
\\
&z_{ijs}\le y_i
&&\forall i\in I,\ j\in J,\ s\in S,
\\
&y_i\in\{0,1\},\quad z_{ijs}\in\{0,1\},\quad x_{ijks}\ge 0.
\end{aligned}
$$

The capacity constraint is written for product \(k=0\), which is the only product in this example. The demand-dependent upper bound in the assignment-linking constraint is valid because no shipment to customer \(j\) can exceed that customer's total demand in scenario \(s\). Together with the exactly-one assignment constraint and demand balance, it ensures that a customer's demand is supplied by its assigned facility. In the code, these constraints appear as `Demand`, `Capacity`, `Assignment`, `Single_Assignment`, and `Facility_Operation` constraints, respectively.

A mixed-integer solver searches over binary decisions and continuous shipments. Such models can require exponential time in the worst case as the number of discrete decisions grows; an optimal solver status certifies optimality for the modeled instance subject to the solver's numerical tolerances. Multiple optimal solutions may exist, so the selected facility or assignment pattern need not be unique.

### Solution sequence

1. Define the input data and check its dimensions and probability assumptions.
2. Create binary opening and assignment variables and nonnegative shipment variables.
3. Add the expected-cost objective and all demand, capacity, assignment, and facility-operation constraints.
4. Solve the integer program, extract the decisions, and verify that the reported cost components agree with the objective.
5. Visualize the solution and examine how demand and capacity scaling affect feasibility and cost.

This sequence implements the mathematical formulation directly using PuLP and the CBC solver through PuLP's `COIN_CMD` interface. The CBC command-line executable must be available to this interface. The numerical example uses fixed costs \([100,120,90]\), capacities \([80,90,70]\), high-scenario demands \([50,40,35,45]\), low-scenario demands \([30,25,20,25]\), and probabilities \(0.5\) for each scenario. Transportation costs are specified in the data cell.

### Interpreting the results

The opening decisions identify the facilities used in every scenario. The assignment and shipment reports show how service responds to each demand scenario. The cost breakdown separates fixed cost from probability-weighted variable cost; scenario transportation costs are also reported before probability weighting. Capacity utilization is calculated as shipped quantity divided by the capacity of an open facility. Sensitivity results show the effect of the specified demand and capacity multipliers; cases without a solver-reported optimal solution are not treated as cost observations.

In [ ]:
from dataclasses import dataclass
from math import isclose
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pulp import (
    COIN_CMD,
    LpMinimize,
    LpProblem,
    LpVariable,
    lpSum,
)
from pulp.constants import LpStatus, LpStatusOptimal

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class SupplyChainInstance:
    """Input data for the scenario-based supply chain design model."""

    facilities: List[int]
    customers: List[int]
    scenarios: List[int]
    fixed_costs: List[float]
    demands: Dict[Tuple[int, int, int], float]
    capacities: List[float]
    transport_costs: Dict[Tuple[int, int], float]
    scenario_probs: List[float]


def create_example_instance() -> SupplyChainInstance:
    """Create the three-facility, four-customer example instance."""
    facilities = [0, 1, 2]
    customers = [0, 1, 2, 3]
    scenarios = [0, 1]

    fixed_costs = [100, 120, 90]
    capacities = [80, 90, 70]
    scenario_probs = [0.5, 0.5]

    transport_costs = {
        (0, 0): 2.5,
        (0, 1): 3.0,
        (0, 2): 3.5,
        (0, 3): 4.0,
        (1, 0): 3.2,
        (1, 1): 2.1,
        (1, 2): 4.1,
        (1, 3): 3.6,
        (2, 0): 4.1,
        (2, 1): 4.6,
        (2, 2): 2.0,
        (2, 3): 2.4,
    }

    # The demand key is (customer, product, scenario); the example has product 0.
    demands = {}
    high_demand = [50, 40, 35, 45]
    for customer, demand in enumerate(high_demand):
        demands[(customer, 0, 0)] = demand

    low_demand = [30, 25, 20, 25]
    for customer, demand in enumerate(low_demand):
        demands[(customer, 0, 1)] = demand

    instance = SupplyChainInstance(
        facilities=facilities,
        customers=customers,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        demands=demands,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs,
    )

    assert len(instance.fixed_costs) == len(instance.facilities)
    assert len(instance.capacities) == len(instance.facilities)
    assert len(instance.scenario_probs) == len(instance.scenarios)
    assert isclose(sum(instance.scenario_probs), 1.0, rel_tol=1e-9, abs_tol=1e-9)
    assert all(probability >= 0 for probability in instance.scenario_probs)
    assert all(
        (customer, 0, scenario) in instance.demands
        for customer in instance.customers
        for scenario in instance.scenarios
    )
    assert all(
        (facility, customer) in instance.transport_costs
        for facility in instance.facilities
        for customer in instance.customers
    )

    return instance


instance = create_example_instance()
print(f"Facilities: {instance.facilities}")
print(f"Customers: {instance.customers}")
print(f"Scenarios: {instance.scenarios} (0 = high demand, 1 = low demand)")
print(f"Fixed costs: {instance.fixed_costs}")
print(f"Capacities: {instance.capacities}")
print(f"Scenario probabilities: {instance.scenario_probs}")

In [ ]:
def solve_supply_chain_network_design(instance: SupplyChainInstance):
    """Build and solve the scenario-based integer programming model."""
    prob = LpProblem("Supply_Chain_Network_Design", LpMinimize)

    # Opening decisions are shared across scenarios, as they are made first.
    y = {
        facility: LpVariable(f"y_{facility}", cat="Binary")
        for facility in instance.facilities
    }

    # The example contains one product, indexed by 0.
    products = [0]
    x = {
        (facility, customer, product, scenario): LpVariable(
            f"x_{facility}_{customer}_{product}_{scenario}", lowBound=0
        )
        for facility in instance.facilities
        for customer in instance.customers
        for product in products
        for scenario in instance.scenarios
    }

    # Assignments are recourse decisions and may differ by scenario.
    z = {
        (facility, customer, scenario): LpVariable(
            f"z_{facility}_{customer}_{scenario}", cat="Binary"
        )
        for facility in instance.facilities
        for customer in instance.customers
        for scenario in instance.scenarios
    }

    fixed_cost_component = lpSum(
        instance.fixed_costs[facility] * y[facility]
        for facility in instance.facilities
    )
    expected_variable_cost_component = lpSum(
        instance.scenario_probs[scenario]
        * instance.transport_costs[(facility, customer)]
        * x[(facility, customer, 0, scenario)]
        for facility in instance.facilities
        for customer in instance.customers
        for scenario in instance.scenarios
    )
    prob += (
        fixed_cost_component + expected_variable_cost_component,
        "Total_Expected_Cost",
    )

    # Each customer's demand is fully supplied in every scenario.
    for customer in instance.customers:
        for product in products:
            for scenario in instance.scenarios:
                prob += (
                    lpSum(
                        x[(facility, customer, product, scenario)]
                        for facility in instance.facilities
                    )
                    == instance.demands[(customer, product, scenario)],
                    f"Demand_{customer}_{product}_{scenario}",
                )

    # Capacity is available only at an open facility and applies scenario by scenario.
    for facility in instance.facilities:
        for scenario in instance.scenarios:
            prob += (
                lpSum(
                    x[(facility, customer, 0, scenario)]
                    for customer in instance.customers
                )
                <= instance.capacities[facility] * y[facility],
                f"Capacity_{facility}_{scenario}",
            )

    # This demand-based bound links shipment quantities to binary assignments.
    for facility in instance.facilities:
        for customer in instance.customers:
            for product in products:
                for scenario in instance.scenarios:
                    prob += (
                        x[(facility, customer, product, scenario)]
                        <= instance.demands[(customer, product, scenario)]
                        * z[(facility, customer, scenario)],
                        f"Assignment_{facility}_{customer}_{product}_{scenario}",
                    )

    # Exactly one facility serves each customer in each scenario.
    for customer in instance.customers:
        for scenario in instance.scenarios:
            prob += (
                lpSum(
                    z[(facility, customer, scenario)]
                    for facility in instance.facilities
                )
                == 1,
                f"Single_Assignment_{customer}_{scenario}",
            )

    # A facility can be assigned customers only if its first-stage decision opens it.
    for facility in instance.facilities:
        for customer in instance.customers:
            for scenario in instance.scenarios:
                prob += (
                    z[(facility, customer, scenario)] <= y[facility],
                    f"Facility_Operation_{facility}_{customer}_{scenario}",
                )

    print("Solving the supply chain network design problem...")
    prob.solve(COIN_CMD(msg=False))

    return prob, y, x, z


prob, y_vars, x_vars, z_vars = solve_supply_chain_network_design(instance)
print(f"Problem status: {LpStatus[prob.status]}")
if prob.status == LpStatusOptimal:
    print(f"Optimal objective value: ${prob.objective.value():,.2f}")
else:
    print("An optimal solution was not reported by the solver.")

In [ ]:
def extract_solution(instance, prob, y_vars, x_vars, z_vars):
    """Extract decisions and independently calculate their cost components."""
    if prob.status != LpStatusOptimal:
        return None

    opened_facilities = [
        facility
        for facility in instance.facilities
        if y_vars[facility].value() is not None and y_vars[facility].value() > 0.5
    ]

    assignments = {}
    shipments = {}
    for scenario in instance.scenarios:
        assignments[scenario] = {}
        shipments[scenario] = {}

        for customer in instance.customers:
            assigned_facilities = [
                facility
                for facility in instance.facilities
                if z_vars[(facility, customer, scenario)].value() is not None
                and z_vars[(facility, customer, scenario)].value() > 0.5
            ]
            if len(assigned_facilities) != 1:
                raise AssertionError(
                    "An optimal solution must assign each customer to exactly one facility."
                )

            facility = assigned_facilities[0]
            assignments[scenario][customer] = facility
            quantity = x_vars[(facility, customer, 0, scenario)].value()
            shipments[scenario][(facility, customer)] = quantity

    fixed_cost = sum(instance.fixed_costs[facility] for facility in opened_facilities)
    variable_costs = {}
    for scenario in instance.scenarios:
        variable_costs[scenario] = sum(
            instance.transport_costs[(facility, customer)] * quantity
            for (facility, customer), quantity in shipments[scenario].items()
        )

    total_variable_cost = sum(
        instance.scenario_probs[scenario] * variable_costs[scenario]
        for scenario in instance.scenarios
    )
    total_cost = fixed_cost + total_variable_cost

    # These checks connect the reported plan and cost breakdown to the model.
    for scenario in instance.scenarios:
        for customer in instance.customers:
            facility = assignments[scenario][customer]
            quantity = shipments[scenario][(facility, customer)]
            assert isclose(
                quantity,
                instance.demands[(customer, 0, scenario)],
                rel_tol=1e-7,
                abs_tol=1e-7,
            )
            assert facility in opened_facilities

        for facility in opened_facilities:
            total_shipment = sum(
                shipments[scenario].get((facility, customer), 0.0)
                for customer in instance.customers
            )
            assert total_shipment <= instance.capacities[facility] + 1e-7

    assert isclose(
        total_cost,
        prob.objective.value(),
        rel_tol=1e-7,
        abs_tol=1e-7,
    )

    return {
        "opened_facilities": opened_facilities,
        "assignments": assignments,
        "shipments": shipments,
        "fixed_cost": fixed_cost,
        "variable_costs": variable_costs,
        "total_variable_cost": total_variable_cost,
        "total_cost": total_cost,
    }


solution = extract_solution(instance, prob, y_vars, x_vars, z_vars)
if solution is not None:
    print("\nOptimal solution")
    print(f"Opened facilities: {solution['opened_facilities']}")
    print(f"Fixed cost: ${solution['fixed_cost']:,.2f}")
    print(f"Expected variable cost: ${solution['total_variable_cost']:,.2f}")
    print(f"Total expected cost: ${solution['total_cost']:,.2f}")

    print("\nCustomer assignments by scenario:")
    for scenario in instance.scenarios:
        scenario_name = "High demand" if scenario == 0 else "Low demand"
        print(
            f"\n{scenario_name} scenario "
            f"(probability = {instance.scenario_probs[scenario]:.2f}):"
        )
        for customer in instance.customers:
            facility = solution["assignments"][scenario][customer]
            quantity = solution["shipments"][scenario][(facility, customer)]
            cost = instance.transport_costs[(facility, customer)] * quantity
            print(
                f"  Customer {customer} -> Facility {facility}: "
                f"{quantity:.0f} units; transportation cost ${cost:,.2f}"
            )
        print(
            f"  Scenario transportation cost: "
            f"${solution['variable_costs'][scenario]:,.2f}"
        )
else:
    print("No optimal solution is available to report.")

In [ ]:
def visualize_network_solution(instance, solution):
    """Plot assignments, expected costs, capacity use, and scenario costs."""
    if solution is None:
        print("No optimal solution is available to visualize.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Supply Chain Network Design: Integer Programming Results",
        fontsize=16,
        fontweight="bold",
    )

    # Draw the high-demand assignments with a simple facility-customer network.
    ax1 = axes[0, 0]
    facility_positions = {
        facility: (index * 2, 2)
        for index, facility in enumerate(instance.facilities)
    }
    customer_positions = {
        customer: (index * 2 + 1, 0)
        for index, customer in enumerate(instance.customers)
    }

    for customer in instance.customers:
        facility = solution["assignments"][0][customer]
        start = facility_positions[facility]
        end = customer_positions[customer]
        ax1.plot(
            [start[0], end[0]],
            [start[1], end[1]],
            color="gray",
            linewidth=2,
            zorder=1,
        )

    for facility in instance.facilities:
        position = facility_positions[facility]
        is_open = facility in solution["opened_facilities"]
        ax1.scatter(
            position[0],
            position[1],
            s=800,
            color="green" if is_open else "red",
            edgecolor="black",
            zorder=2,
            label="Open facility" if is_open else "Closed facility",
        )
        ax1.text(*position, f"F{facility}", ha="center", va="center", zorder=3)

    for customer in instance.customers:
        position = customer_positions[customer]
        ax1.scatter(
            position[0],
            position[1],
            s=600,
            color="royalblue",
            edgecolor="black",
            zorder=2,
            label="Customer" if customer == instance.customers[0] else None,
        )
        ax1.text(*position, f"C{customer}", ha="center", va="center", zorder=3)

    handles, labels = ax1.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    ax1.legend(unique_labels.values(), unique_labels.keys(), loc="best")
    ax1.set_title("Assignments: High-Demand Scenario")
    ax1.set_xlim(-1, max(len(instance.facilities) * 2, len(instance.customers) * 2))
    ax1.set_ylim(-0.7, 2.7)
    ax1.axis("off")

    # The expected variable-cost component includes scenario probabilities.
    ax2 = axes[0, 1]
    cost_components = ["Fixed cost", "Expected variable cost"]
    cost_values = [
        solution["fixed_cost"],
        solution["total_variable_cost"],
    ]
    bars = ax2.bar(cost_components, cost_values, color=["#FF6B6B", "#4ECDC4"])
    ax2.set_ylabel("Cost ($)")
    ax2.set_title("Expected-Cost Breakdown")
    ax2.grid(True, axis="y", alpha=0.3)
    label_offset = max(cost_values) * 0.01 if max(cost_values) > 0 else 0.01
    for bar, value in zip(bars, cost_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + label_offset,
            f"${value:,.0f}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    # Utilization is measured against the capacity of each open facility.
    ax3 = axes[1, 0]
    opened_facilities = solution["opened_facilities"]
    positions = np.arange(len(opened_facilities))
    width = 0.35
    for scenario, label, color, offset in [
        (0, "High demand", "#FF6B6B", -width / 2),
        (1, "Low demand", "#4ECDC4", width / 2),
    ]:
        utilization = []
        for facility in opened_facilities:
            shipped = sum(
                solution["shipments"][scenario].get((facility, customer), 0.0)
                for customer in instance.customers
            )
            utilization.append(100 * shipped / instance.capacities[facility])
        ax3.bar(
            positions + offset,
            utilization,
            width,
            label=label,
            color=color,
            alpha=0.85,
        )

    ax3.set_xlabel("Open facility")
    ax3.set_ylabel("Capacity utilization (%)")
    ax3.set_title("Capacity Utilization by Scenario")
    ax3.set_xticks(positions)
    ax3.set_xticklabels([f"F{facility}" for facility in opened_facilities])
    ax3.legend()
    ax3.grid(True, axis="y", alpha=0.3)

    # These bars show scenario costs before probability weighting.
    ax4 = axes[1, 1]
    scenario_names = [
        "High demand" if scenario == 0 else "Low demand"
        for scenario in instance.scenarios
    ]
    scenario_costs = [
        solution["variable_costs"][scenario]
        for scenario in instance.scenarios
    ]
    bars = ax4.bar(
        scenario_names,
        scenario_costs,
        color=["#FF6B6B", "#4ECDC4"],
    )
    ax4.set_ylabel("Transportation cost ($)")
    ax4.set_title("Transportation Cost by Scenario")
    ax4.grid(True, axis="y", alpha=0.3)
    label_offset = max(scenario_costs) * 0.01 if max(scenario_costs) > 0 else 0.01
    for bar, cost, scenario in zip(bars, scenario_costs, instance.scenarios):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + label_offset,
            f"${cost:,.0f}\np={instance.scenario_probs[scenario]:.2f}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    plt.tight_layout()
    plt.show()


visualize_network_solution(instance, solution)

In [ ]:
def sensitivity_analysis(instance):
    """Solve the same model under demand and capacity scaling experiments."""
    print("\nSensitivity analysis")

    demand_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    demand_results = []
    unreported_demand_multipliers = []

    print("\nDemand sensitivity:")
    for multiplier in demand_multipliers:
        modified_instance = create_example_instance()
        modified_instance.demands = {
            key: demand * multiplier
            for key, demand in modified_instance.demands.items()
        }

        modified_prob, modified_y, modified_x, modified_z = (
            solve_supply_chain_network_design(modified_instance)
        )
        modified_solution = extract_solution(
            modified_instance,
            modified_prob,
            modified_y,
            modified_x,
            modified_z,
        )

        if modified_solution is None:
            unreported_demand_multipliers.append(multiplier)
            continue

        demand_results.append(
            {
                "Demand_Multiplier": multiplier,
                "Total_Cost": modified_solution["total_cost"],
                "Opened_Facilities": len(
                    modified_solution["opened_facilities"]
                ),
            }
        )

    demand_df = pd.DataFrame(demand_results)
    if not demand_df.empty:
        print(demand_df.to_string(index=False))
    if unreported_demand_multipliers:
        print(
            "No optimal solution reported for demand multipliers: "
            f"{unreported_demand_multipliers}"
        )

    capacity_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    capacity_results = []
    unreported_capacity_multipliers = []

    print("\nCapacity sensitivity:")
    for multiplier in capacity_multipliers:
        modified_instance = create_example_instance()
        modified_instance.capacities = [
            capacity * multiplier
            for capacity in modified_instance.capacities
        ]

        modified_prob, modified_y, modified_x, modified_z = (
            solve_supply_chain_network_design(modified_instance)
        )
        modified_solution = extract_solution(
            modified_instance,
            modified_prob,
            modified_y,
            modified_x,
            modified_z,
        )

        if modified_solution is None:
            unreported_capacity_multipliers.append(multiplier)
            continue

        capacity_results.append(
            {
                "Capacity_Multiplier": multiplier,
                "Total_Cost": modified_solution["total_cost"],
                "Opened_Facilities": len(
                    modified_solution["opened_facilities"]
                ),
            }
        )

    capacity_df = pd.DataFrame(capacity_results)
    if not capacity_df.empty:
        print(capacity_df.to_string(index=False))
    if unreported_capacity_multipliers:
        print(
            "No optimal solution reported for capacity multipliers: "
            f"{unreported_capacity_multipliers}"
        )

    # Plot only solver-reported optimal solutions; other cases are not costs.
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    if not demand_df.empty:
        axes[0].plot(
            demand_df["Demand_Multiplier"],
            demand_df["Total_Cost"],
            "o-",
            linewidth=2,
            markersize=8,
            color="#FF6B6B",
        )
    axes[0].set_xlabel("Demand multiplier")
    axes[0].set_ylabel("Total expected cost ($)")
    axes[0].set_title("Demand Sensitivity")
    axes[0].set_xticks(demand_multipliers)
    axes[0].grid(True, alpha=0.3)

    if not capacity_df.empty:
        axes[1].plot(
            capacity_df["Capacity_Multiplier"],
            capacity_df["Total_Cost"],
            "o-",
            linewidth=2,
            markersize=8,
            color="#4ECDC4",
        )
    axes[1].set_xlabel("Capacity multiplier")
    axes[1].set_ylabel("Total expected cost ($)")
    axes[1].set_title("Capacity Sensitivity")
    axes[1].set_xticks(capacity_multipliers)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return demand_df, capacity_df


demand_sensitivity, capacity_sensitivity = sensitivity_analysis(instance)

## Interpreting the model's results

The integer program makes the facility-opening decision jointly with scenario-specific customer assignments and shipments. A facility's fixed cost is incurred once if it is opened, while transportation cost changes with the scenario's demand and assignment decisions. The objective compares these cost components on a common expected-cost basis.

The assignment constraint is intentionally restrictive: each customer is served by exactly one facility in each scenario. The shipment variables therefore report the full customer demand at that assigned facility. Capacity utilization can differ between scenarios even though the open facilities do not, because demand and assignments are scenario-dependent.

The sensitivity analysis resolves the same mathematical model after scaling demand or capacity. A case contributes a cost observation only when the solver reports an optimal solution; a case without a solver-reported optimal solution has no valid optimal cost to plot. Changes in total expected cost reflect both shipment quantities and any changes in the facility-opening decisions.

### Strengths and limitations

- **Scenario representation:** Known discrete scenarios and probabilities make demand uncertainty explicit in the expected-cost objective.
- **Decision timing:** Shared facility-opening variables represent decisions made before the demand scenario is known; assignment and shipment variables represent scenario-dependent recourse.
- **Optimization guarantee:** The solver can certify an optimal solution for this finite mixed-integer model when it reports optimal status, subject to numerical tolerances.
- **Computational growth:** Mixed-integer optimization can become computationally demanding as the numbers of facilities, customers, and scenarios grow. The number of binary opening and assignment decisions increases with the instance size, and worst-case solution time can grow exponentially.
- **Data and scope:** The model requires costs, capacities, demands, and scenario probabilities. It represents uncertainty through the specified discrete scenarios and does not include operational features outside the stated assumptions.

The formulation is a useful exact benchmark for small instances and for strategic decisions where scenario-based expected cost and optimality certification are important.